# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shiva-sn/ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

**Lane:** Structured Content Archetype Clustering

This notebook frames the capstone as an unsupervised ML task and connects the task definition to the actual content-level data and later clustering pipeline.

## 1. My Lane as an ML Task (Type)

**Task type: clustering.** The main question is: *what recurring content archetypes exist in the inventory based on observed search demand, visibility, freshness, content size, and engagement?* There is no known target label to predict. Clustering is therefore more appropriate than classification. The later action queue may rank review work, but that ranking is a decision-support layer after the clusters are formed, not the ML task being framed here.

The downstream W05 implementation uses K-Means and three clusters. Cluster labels are discovered groups, not ground-truth classes.

In [ ]:
task_type = "clustering"
has_target_label = False
model_family = "K-Means"
final_k = 3

print("Task type:", task_type)
print("Target label present:", has_target_label)
print("Model family used in W05:", model_family)
print("Reference number of clusters:", final_k)

assert task_type == "clustering"
assert has_target_label is False
assert model_family == "K-Means"
assert final_k >= 2
print("Task-framing checks: PASS")

## 2. Target or Proxy

There is **no target or proxy target** for the clustering task. The eight core features are descriptive inputs only: `search_volume`, `word_count`, `content_age_days`, `days_since_update`, `impressions_90d`, `ctr_90d`, `avg_position_90d`, and `engagement_rate`.

The cluster assignment is produced by the model itself. It is not a hand-written label such as "stale" or "high performer". Those descriptive names are assigned only after the cluster profiles are inspected. Future outcomes and trend labels are excluded from the core feature matrix.

In [ ]:
core_features = [
    "search_volume",
    "word_count",
    "content_age_days",
    "days_since_update",
    "impressions_90d",
    "ctr_90d",
    "avg_position_90d",
    "engagement_rate",
]

target_column = None
proxy_target_columns = []

print("Target column:", target_column)
print("Proxy target columns:", proxy_target_columns)
print("Core feature count:", len(core_features))

assert target_column is None
assert proxy_target_columns == []
assert len(core_features) == 8
assert "client_hash_id" not in core_features
assert "content_hash_id" not in core_features
assert not any(k in c.lower() for c in core_features for k in ["future", "trend", "label", "target", "outcome"])
print("Target/proxy checks: PASS")

## 3. Success metric

For an unsupervised model, **silhouette score** is the primary defensible internal metric because there is no ground-truth archetype label. A higher silhouette indicates that observations are, on average, more separated from other clusters relative to their own cluster.

It is not accuracy, business impact, or proof that the clusters are true categories. The score must be considered together with cluster size, stability, interpretability, and decision usefulness. W05 uses the silhouette score as the main separation metric while checking cluster balance.

In [ ]:
success_metric = "silhouette_score"
reference_silhouette = 0.8414

print("Primary success metric:", success_metric)
print("W05 reference silhouette:", reference_silhouette)

assert success_metric == "silhouette_score"
assert 0 <= reference_silhouette <= 1
print("Metric framing check: PASS")

## 4. The unit of analysis, as a real dataframe

**One row = one content item** in the content-level modeling dataset. The W05 pipeline combines content attributes with aggregated 90-day search/performance signals and creates a one-row-per-content `model_df`. The observation window for performance is 90 days, with content age and days since update measured relative to the analysis snapshot.

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path("..")
candidates = [
    BASE / "outputs" / "content_archetypes_clustered.parquet",
    BASE / "outputs" / "content_level_model_dataset.parquet",
    BASE / "data" / "raw" / "content_refresh_anonymized.csv",
    BASE / "data" / "raw" / "content_refresh_anonymized.parquet",
]

DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("No supported content dataset found. Run W05 or place the anonymized starter data under data/raw/.")

if DATA_PATH.suffix.lower() == ".csv":
    content_df = pd.read_csv(DATA_PATH)
else:
    content_df = pd.read_parquet(DATA_PATH)

print("Loaded:", DATA_PATH.resolve())
print("Shape:", content_df.shape)

id_candidates = [c for c in ["content_id", "content_hash_id"] if c in content_df.columns]
if not id_candidates:
    raise ValueError("No content identifier found in the selected dataset.")
id_col = id_candidates[0]

print("Content identifier:", id_col)
print("Rows:", len(content_df))
print("Unique content items:", content_df[id_col].nunique())
print("One row per content:", content_df[id_col].nunique() == len(content_df))

assert len(content_df) > 0
assert content_df[id_col].nunique() == len(content_df), "Expected one row per content item for the W02 framing dataset."

available_core = [c for c in core_features if c in content_df.columns]
print("Core features available in this source:", available_core)
print("Available core feature count:", len(available_core))

## 5. Why ML beats a fixed rule here

A single if-statement cannot express the portfolio's mixed patterns cleanly. Content can simultaneously have high search demand, older age, strong engagement, weak position, low impressions, or combinations of these signals. A fixed threshold would force hand-written boundaries onto a multidimensional inventory.

Unsupervised clustering is useful here because it can group items by their joint observed feature profiles first. Human interpretation then gives the resulting groups practical names and actions. This is still descriptive: the clusters organize observed patterns; they do not prove what content changes will cause better future performance.

In [ ]:
# Simple evidence check: compare how many distinct above/below-median combinations exist
# across the eight signals. A high count illustrates why one threshold is too coarse.

available = [c for c in core_features if c in content_df.columns]
if len(available) == len(core_features):
    numeric = content_df[available].apply(pd.to_numeric, errors="coerce")
    medians = numeric.median()
    directional = numeric.ge(medians)
    unique_profiles = directional.dropna(how="all").drop_duplicates().shape[0]
    print("Distinct median-side feature profiles observed:", unique_profiles)
    print("A single threshold on one feature cannot represent these joint patterns.")
else:
    print("The selected source is an aggregated/raw input rather than the final W05 table;")
    print("the fixed-rule comparison is qualitative here and the full feature matrix is built downstream.")

assert task_type == "clustering"
print("ML-vs-rule framing check: PASS")

## Final framing

The capstone is framed as **unsupervised K-Means clustering at one row per content item**. The eight core features describe observed content, freshness, search-performance, and engagement signals. Silhouette score is the primary internal metric, supplemented by cluster balance, stability, interpretability, and manual review. The downstream W07 action queue is a separate decision-support layer.

This framing deliberately avoids causal language: the model discovers observed patterns; it does not establish that an action will increase traffic, rankings, or engagement.

## Self-check

Before submission, confirm that the framing stays aligned with the actual modeling pipeline.

In [ ]:
checks = [
    ("Task is clustering", task_type == "clustering"),
    ("No target label", target_column is None),
    ("Exactly eight core features", len(core_features) == 8),
    ("No identifier feature leakage", not any(c in core_features for c in ["client_hash_id", "content_hash_id"])),
    ("Silhouette used as primary metric", success_metric == "silhouette_score"),
    ("Reference K is defined", final_k == 3),
    ("Source dataset loaded", len(content_df) > 0),
    ("One row per content in selected source", content_df[id_col].nunique() == len(content_df)),
]

check_df = pd.DataFrame(checks, columns=["check", "passed"])
display(check_df)

if not check_df["passed"].all():
    raise AssertionError("One or more W02 framing checks failed.")

print("All W02 framing checks PASS.")